# Text Analytics Coursework

This notebook provides some example code for loading and examining the dataset for task 1. 

In [1]:
%load_ext autoreload
%autoreload 2

# Use HuggingFace's datasets library to access the Emotion dataset
from datasets import load_dataset
import numpy as np
import pandas as pd

# Task 1 - PubMedQA

Steps:
* Import the dataset
* Medical QA pipeline: baseline system
* Evaluation -- some simple metrics as an example; a naive baseline as input to the example

In [2]:
# Load PubMedQA (the main labeled split is "pqa_labeled"). 
# The other splits are "pqa_unlabeled", which has no yes/no/maybe labels, 
# and "pqa_artificial", which consists of automatically generated 
# questions and answers. 
dataset = load_dataset("pubmed_qa", "pqa_labeled")

# Inspect the dataset
print(dataset)

# Look at the first example
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 1000
    })
})
{'pubid': 21645374, 'question': 'Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?', 'context': {'contexts': ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.', 'The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole wi

To access the important fields:

In [3]:
example = dataset["train"][0]

question = example["question"]
context_paragraphs = example["context"]["contexts"]
label = example["final_decision"]

print("Question:", question)
print("Label:", label)
print("What is inside the context? These are the keys to the context dictionary:", list(example["context"].keys()))
print("The 'context' is an abstract that has been split into paragraphs. \n First context paragraph:", context_paragraphs[0])

Question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
Label: yes
What is inside the context? These are the keys to the context dictionary: ['contexts', 'labels', 'meshes', 'reasoning_required_pred', 'reasoning_free_pred']
The 'context' is an abstract that has been split into paragraphs. 
 First context paragraph: Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.


Combine context paragraphs (common preprocessing step)

In [4]:
def build_context(example):
    example["combined_context"] = " ".join(example["context"]["contexts"])
    return example

dataset = dataset.map(build_context)

print(dataset["train"][0]["combined_context"][:500])

Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has b


The short answers are stored in the field "final_decision" and can take the following values: 

In [5]:
np.unique(dataset["train"]["final_decision"])

array(['maybe', 'no', 'yes'], dtype='<U5')

The long answers look like this and your system could either extract the answer sentences from the context or generate them. Producing long answers is optional.

In [6]:
dataset["train"][0]["long_answer"]

'Results depicted mitochondrial dynamics in vivo as PCD progresses within the lace plant, and highlight the correlation of this organelle with other organelles during developmental PCD. To the best of our knowledge, this is the first report of mitochondria and chloroplasts moving on transvacuolar strands to form a ring structure surrounding the nucleus during developmental PCD. Also, for the first time, we have shown the feasibility for the use of CsA in a whole plant system. Overall, our findings implicate the mitochondria as playing a critical and early role in developmentally regulated PCD in the lace plant.'